# 01 — Exploratory Data Analysis

First look at the data in `data/ufc.db` (run `uv run python -m octagon build-db` first).

Questions this notebook answers:
1. How much data do we have, and over what period?
2. Where do we have betting odds (the strongest baseline)?
3. **What is the favorite baseline we must beat?**
4. How much is missing (reach, DOB, stats) — and for whom?

In [ ]:
import math
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

from octagon.betting.odds import implied_probability, remove_vig
from octagon.config import resolve_path

conn = sqlite3.connect(resolve_path("data/ufc.db"))
fights = pd.read_sql("SELECT * FROM fights", conn, parse_dates=["date"])
odds = pd.read_sql("SELECT * FROM odds", conn)
fighters = pd.read_sql("SELECT * FROM fighters", conn)
print(f"{len(fights)} fights, {len(fighters)} fighters, {len(odds)} fights with odds")
fights.head(3)

In [ ]:
# Fight volume over time, and where odds coverage begins.
fights["year"] = fights["date"].dt.year
with_odds = fights.merge(odds, on="fight_id")

fig, ax = plt.subplots(figsize=(10, 4))
fights.groupby("year").size().plot(ax=ax, label="all fights")
with_odds.groupby("year").size().plot(ax=ax, label="fights with odds")
ax.set_ylabel("fights per year")
ax.legend()
ax.set_title("Odds coverage starts ~2014 — the betting sim can only run there")
plt.show()

In [ ]:
# THE number to beat: 'always pick the closing-odds favorite'.
# We also compute the market's log loss using vig-free probabilities —
# our model's log loss must at least approach it to be interesting.
decisive = with_odds[with_odds["winner"].isin(["fighter1", "fighter2"])].copy()

fair = decisive.apply(
    lambda r: remove_vig(implied_probability(r["fighter1_odds"]),
                         implied_probability(r["fighter2_odds"])),
    axis=1, result_type="expand")
decisive[["fair1", "fair2"]] = fair

decisive["fav_correct"] = (
    (decisive["fair1"] >= decisive["fair2"]) == (decisive["winner"] == "fighter1"))
decisive["market_logloss"] = -decisive.apply(
    lambda r: math.log(r["fair1"] if r["winner"] == "fighter1" else r["fair2"]), axis=1)

print(f"favorite accuracy: {decisive['fav_correct'].mean():.4f}  (n={len(decisive)})")
print(f"market log loss:   {decisive['market_logloss'].mean():.4f}")
decisive.groupby("year")["fav_correct"].agg(["mean", "size"])

In [ ]:
# Missing tale-of-the-tape: reach is the big offender. Debuting/obscure
# fighters lack data — exactly the fights models get wrong.
cols = ["height_cm", "reach_cm", "weight_kg", "dob", "stance"]
fighters[cols].isna().mean().round(3).to_frame("missing fraction")

In [ ]:
# Weight class mix and finish methods — sanity that parsing worked.
display(fights["weight_class"].value_counts().head(15))
fights["method"].value_counts().head(10)

## Takeaways

- Odds exist for ~3.4k fights (2014→2023). The **favorite baseline is ~65% accuracy, ~0.62 market log loss** — treat any model claiming much more than that with suspicion.
- Odds are betmma.tips *best-available-line* prices: median overround ~1.01 and 24% of fights sum below 1.0. Normalized (vig-free) probabilities are fine; simulated backtest ROI will be optimistic vs. any single book and must say so.
- Source fighter ordering correlates with the winner (Greco lists winners first ~63% of the time) — the feature builder applies seeded corner swaps, and `python -m octagon audit` verifies antisymmetry.
- Reach/DOB missingness is concentrated in old-era and debuting fighters; models get a missing-data story (NaN-native GBMs, train-only imputation for linear models), not silent zeros.